In [33]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,ID,Age,Race,Marital Status,T Stage,N Stage,6th Stage,differentiate,Grade,A Stage,...,BMI,Heart Rate,Serum Creatinine,Uric Acid,Hemoglobin,GFR,Serum Sodium,Serum Potassium,Serum Albumin,Lactate
0,1,62.300654,White,Widowed,T1,N1,NaN,Moderately differentiated,2,Regional,...,25.228195,98.865449,4.343779,3.781804,13.704187,112.038394,136.560377,-1.506035,4.699045,-3.200633
1,2,37.268422,White,Married,T2,N1,IIB,Well differentiated,1,Regional,...,31.027525,81.547091,-5.053593,1.990754,20.685675,109.605432,147.569841,-0.033068,1.676842,2.013738
2,3,55.864953,White,Single,T1,N1,IIA,Well differentiated,1,Regional,...,20.009729,77.214648,-0.683623,11.299137,10.565341,112.964603,147.176105,-7.076607,6.776799,-7.266369
3,4,60.586799,White,Divorced,T1,N1,IIA,Poorly differentiated,3,Regional,...,33.217567,86.513469,0.317514,0.238220,7.512031,63.477023,135.443017,3.189928,4.519103,-1.637100
4,5,48.197741,White,Separated,T2,N1,IIB,Moderately differentiated,2,Regional,...,33.909838,90.401178,-1.838835,10.500072,19.806571,98.437718,136.071561,8.390279,7.027023,11.230639


In [34]:
df_train.columns

Index(['ID', 'Age', 'Race', 'Marital Status', 'T Stage', 'N Stage',
       '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Tumor Size',
       'Estrogen Status', 'Progesterone Status', 'Regional Node Examined',
       'Reginol Node Positive', 'Status', 'T_N_Stage', 'Hormone_Status',
       'Reginol Node Negative', 'Blood Pressure', 'Diastolic Pressure',
       'Cholesterol', 'Body Temperature', 'Oxygen Saturation',
       'Respiratory Rate', 'Blood Glucose', 'BMI', 'Heart Rate',
       'Serum Creatinine', 'Uric Acid', 'Hemoglobin', 'GFR', 'Serum Sodium',
       'Serum Potassium', 'Serum Albumin', 'Lactate'],
      dtype='object')

In [35]:
#subtasks

subtask1 = []
subtask3 = []
mediana = df_train.BMI.median()
def subtasks(row):
    if row['GFR'] >= 90:
        subtask1.append("Normal")
    else:
        subtask1.append("Mildly Decreased")
    if row['BMI'] > mediana:
        subtask3.append(1)
    else:
        subtask3.append(0)
    return row

df_test = df_test.apply(subtasks, axis=1)

q1 = df_train["Serum Creatinine"].quantile(0.25)
q2 = df_train["Serum Creatinine"].quantile(0.50)
q3 = df_train["Serum Creatinine"].quantile(0.75)

subtask2 = []
def f2(val):
    if val <= q1:
        subtask2.append("Very Low")
    elif val <= q2:
        subtask2.append("Low")
    elif val <= q3:
        subtask2.append("High")
    else:
        subtask2.append("Very High")
    return val

df_test["Serum Creatinine"] = df_test["Serum Creatinine"].apply(f2)

subtask4 = df_test['T Stage'].apply(lambda x: len(df_train[df_train['T Stage'] == x]))

subtask_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.ID,
    "answer": subtask1
})

subtask_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.ID,
    "answer": subtask2
})

subtask_3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.ID,
    "answer": subtask3
})

subtask_4 = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": df_test.ID,
    "answer": subtask4
})

df_subtasks = pd.concat([subtask_1, subtask_2, subtask_3, subtask_4])
df_subtasks

,subtaskID,datapointID,answer
0,1,3220,Normal
1,1,3221,Normal
2,1,3222,Normal
3,1,3223,Mildly Decreased
4,1,3224,Mildly Decreased
...,...,...,...
800,4,4020,1281
801,4,4021,1430
802,4,4022,1430
803,4,4023,1281


In [36]:
df_train["6th Stage"].value_counts()

6th Stage
IIA     927
IIB     908
IIIA    830
IIIC    389
IIIB     62
Name: count, dtype: int64

In [37]:
#preprocessing
to_encode = ['T Stage', 'N Stage', '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Estrogen Status', 'Progesterone Status', 'T_N_Stage', 'Hormone_Status', 'Race']
to_impute = ['6th Stage', 'Tumor Size']

df_train["6th Stage"] = df_train["6th Stage"].fillna("IIA")
df_train['Tumor Size'] = df_train['Tumor Size'].fillna(df_train['Tumor Size'].mean())

from sklearn.preprocessing import TargetEncoder

for col in to_encode:
    enc = TargetEncoder()
    df_train[col] = enc.fit_transform(pd.DataFrame(df_train[col]), df_train["Status"])
    df_test[col] = enc.transform(pd.DataFrame(df_test[col]))

In [38]:
features = ['Age', 'T Stage', 'N Stage',
       '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Tumor Size',
       'Estrogen Status', 'Progesterone Status', 'Regional Node Examined',
       'Reginol Node Positive', 'T_N_Stage', 'Hormone_Status',
       'Reginol Node Negative', 'Blood Pressure', 'Diastolic Pressure',
       'Cholesterol', 'Body Temperature', 'Oxygen Saturation',
       'Respiratory Rate', 'Blood Glucose', 'BMI', 'Heart Rate',
       'Serum Creatinine', 'Uric Acid', 'Hemoglobin', 'GFR', 'Serum Sodium',
       'Serum Potassium', 'Serum Albumin', 'Lactate']
target = "Status"

X = df_train[features]
y = df_train[target].map({"Dead": 0, "Alive": 1})

X_final = df_test[features]

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
y.value_counts()

Status
1    2723
0     496
Name: count, dtype: int64

In [47]:
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

model = CatBoostClassifier(random_state=42, silent=True).fit(X_train, y_train)
y_preds = model.predict(X_valid)
print(classification_report(y_valid, y_preds))

              precision    recall  f1-score   support

           0       0.53      0.09      0.15        92
           1       0.87      0.99      0.92       552

    accuracy                           0.86       644
   macro avg       0.70      0.54      0.54       644
weighted avg       0.82      0.86      0.81       644



In [48]:
y_final = model.predict(X_final)

In [49]:
df_preds = pd.DataFrame({
    "subtaskID": 5,
    "datapointID": df_test.ID,
    "answer": y_final
})

df_preds.answer = df_preds.answer.map({1: "Alive", 0:"Dead"})

df_preds.head()

,subtaskID,datapointID,answer
0,5,3220,Alive
1,5,3221,Alive
2,5,3222,Alive
3,5,3223,Alive
4,5,3224,Alive


In [50]:
df_preds.answer.value_counts()

answer
Alive    786
Dead      19
Name: count, dtype: int64

In [51]:
submission = pd.concat([df_subtasks, df_preds])
submission

,subtaskID,datapointID,answer
0,1,3220,Normal
1,1,3221,Normal
2,1,3222,Normal
3,1,3223,Mildly Decreased
4,1,3224,Mildly Decreased
...,...,...,...
800,5,4020,Alive
801,5,4021,Alive
802,5,4022,Alive
803,5,4023,Alive


In [52]:
submission.to_csv("submission.csv", index=False)